# Dataset Profiling EDA

This notebook programmatically scans the dataset directories, extracts audio metadata (like sample rate, duration, channels), and plots the distributions. Use the findings here to populate the markdown documentation in `docs/datasets/`.

> **Running on Colab:** Run the cell below to mount Google Drive and set the correct `DATA_ROOT` path.

In [ ]:
import os

# ── Colab: mount Google Drive ────────────────────────────────────────────────
try:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_ROOT = "/content/drive/MyDrive/DL-Project/data/raw"
    print("Running on Colab. DATA_ROOT =", DATA_ROOT)
except ImportError:
    # Local / non-Colab environment
    DATA_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", "data", "raw"))
    print("Running locally. DATA_ROOT =", DATA_ROOT)

print("DATA_ROOT exists:", os.path.isdir(DATA_ROOT))

In [ ]:
import glob
import wave
import contextlib
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

## Utility Functions

In [ ]:
def extract_audio_metadata_wav(file_path):
    """Extract metadata from a WAV file without fully loading the audio array."""
    try:
        with contextlib.closing(wave.open(file_path, "rb")) as f:
            frames = f.getnframes()
            rate = f.getframerate()
            duration = frames / float(rate)
            channels = f.getnchannels()
            bit_depth = f.getsampwidth() * 8
            return {
                "file": file_path,
                "duration": duration,
                "sample_rate": rate,
                "channels": channels,
                "bit_depth": bit_depth,
                "error": None,
            }
    except Exception as exc:
        return {
            "file": file_path,
            "duration": None,
            "sample_rate": None,
            "channels": None,
            "bit_depth": None,
            "error": str(exc),
        }


def analyze_dataset_directory(dataset_dir, ext="*.wav"):
    """Crawl a directory recursively for files and extract their metadata."""
    files = glob.glob(os.path.join(dataset_dir, f"**/{ext}"), recursive=True)
    print(f"Found {len(files)} files matching '{ext}' in: {dataset_dir}")

    metadata = []
    for f in files:
        if ext == "*.wav":
            metadata.append(extract_audio_metadata_wav(f))
        # Add elif blocks here for other formats (e.g. *.avi) as needed.

    return pd.DataFrame(metadata)


def plot_metadata_distributions(df, dataset_name):
    """Plot duration, sample-rate, and channel-type distributions."""
    if df.empty:
        print(f"[{dataset_name}] DataFrame is empty — nothing to plot.")
        return

    # Rows without errors
    valid_df = df[df["error"].isna()].copy()

    if valid_df.empty:
        print(f"[{dataset_name}] All files had errors — nothing to plot.")
        return

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle(f"{dataset_name} — Audio Profiling", fontsize=16)

    # Duration distribution
    sns.histplot(valid_df["duration"], bins=30, ax=axes[0], kde=True)
    axes[0].set_title("Duration (s)")
    axes[0].set_xlabel("Seconds")

    # Sample-rate counts
    sns.countplot(x="sample_rate", data=valid_df, ax=axes[1])
    axes[1].set_title("Sample Rates (Hz)")

    # Mono vs Stereo
    valid_df["channel_type"] = valid_df["channels"].map({1: "Mono", 2: "Stereo"})
    sns.countplot(x="channel_type", data=valid_df, ax=axes[2])
    axes[2].set_title("Channels")

    plt.tight_layout()
    plt.show()

    # Print a ready-to-paste Markdown summary
    print(f"\n=== {dataset_name} — Markdown Summary ===")
    print(f"- **Total Files:** {len(valid_df)}")
    print(f"- **Total Duration:** {valid_df['duration'].sum() / 3600:.2f} hours")
    print(f"- **Average Clip Length:** {valid_df['duration'].mean():.2f}s")
    print(f"- **Sample Rates:** {valid_df['sample_rate'].value_counts().to_dict()}")
    print(f"- **Channels:** {valid_df['channel_type'].value_counts().to_dict()}")
    if df["error"].notna().any():
        print(f"- **Skipped (errors):** {df['error'].notna().sum()} files")
    print("==========================================\n")

## 1. DESED Analysis
*(Make sure data exists locally or mount Google Drive first)*

In [ ]:
desed_dir = os.path.join(DATA_ROOT, "DESED", "audio")
if os.path.exists(desed_dir):
    df_desed = analyze_dataset_directory(desed_dir, ext="*.wav")
    plot_metadata_distributions(df_desed, "DESED")
else:
    print(f"Directory not found: {desed_dir}")

## 2. CAUCAFall Analysis
Note: Since CAUCAFall is natively `.avi`, you might need to extract audio first using `ffmpeg` before running this, or modify `analyze_dataset_directory` to use MoviePy/FFmpeg.

In [ ]:
# Example implementation stub for analyzing AVI files
caucafall_dir = os.path.join(DATA_ROOT, "CAUCAFall")
if os.path.exists(caucafall_dir):
    df_cauca = analyze_dataset_directory(caucafall_dir, ext="*.avi")
    # Note: df_cauca will show errors unless you implement AVI audio metadata extraction
    # plot_metadata_distributions(df_cauca, "CAUCAFall")
else:
    print(f"Directory not found: {caucafall_dir}")

## 3. WaterLeakage Analysis

In [ ]:
water_dir = os.path.join(DATA_ROOT, "water_leakage_voice_data")
if os.path.exists(water_dir):
    df_water = analyze_dataset_directory(water_dir, ext="*.wav")
    plot_metadata_distributions(df_water, "Water Leakage")
else:
    print(f"Directory not found: {water_dir}")

## 4. Human Screaming Detection Analysis

In [ ]:
screaming_dir = os.path.join(DATA_ROOT, "HumanScreaming")
if os.path.exists(screaming_dir):
    df_screaming = analyze_dataset_directory(screaming_dir, ext="*.wav")
    plot_metadata_distributions(df_screaming, "Human Screaming Detection")

    # Class breakdown from folder names
    df_screaming["label"] = df_screaming["file"].apply(
        lambda p: os.path.basename(os.path.dirname(p))
    )
    print("Class distribution:\n", df_screaming["label"].value_counts())
else:
    print(f"Directory not found: {screaming_dir}")

## 5. FSD50K Analysis
This is a large dataset (~51k files). The loop below uses `wave` for fast metadata extraction without loading audio into memory.

In [ ]:
fsd50k_dir = os.path.join(DATA_ROOT, "FSD50K")
if os.path.exists(fsd50k_dir):
    df_fsd = analyze_dataset_directory(fsd50k_dir, ext="*.wav")
    plot_metadata_distributions(df_fsd, "FSD50K")

    # Show which subdirectories (splits/classes) are present
    df_fsd["split"] = df_fsd["file"].apply(
        lambda p: p.replace(fsd50k_dir, "").lstrip(os.sep).split(os.sep)[0]
    )
    print("Split breakdown:\n", df_fsd["split"].value_counts())
else:
    print(f"Directory not found: {fsd50k_dir}")

## 6. SAFE (Sound Analysis for Fall Events) Analysis

In [ ]:
safe_dir = os.path.join(DATA_ROOT, "SAFE")
if os.path.exists(safe_dir):
    df_safe = analyze_dataset_directory(safe_dir, ext="*.wav")
    plot_metadata_distributions(df_safe, "SAFE")

    # Class breakdown from folder names (Falls vs ADL)
    df_safe["label"] = df_safe["file"].apply(
        lambda p: p.replace(safe_dir, "").lstrip(os.sep).split(os.sep)[0]
    )
    print("Class distribution:\n", df_safe["label"].value_counts())
else:
    print(f"Directory not found: {safe_dir}")

## 7. AudioSet (Extracted Clips) Analysis
Only run this section if you have already downloaded clips via `yt-dlp`.

In [ ]:
audioset_dir = os.path.join(DATA_ROOT, "AudioSet")
if os.path.exists(audioset_dir):
    df_audioset = analyze_dataset_directory(audioset_dir, ext="*.wav")
    plot_metadata_distributions(df_audioset, "AudioSet")

    # Show which sub-class folders were downloaded
    df_audioset["label"] = df_audioset["file"].apply(
        lambda p: p.replace(audioset_dir, "").lstrip(os.sep).split(os.sep)[0]
    )
    print("Downloaded class breakdown:\n", df_audioset["label"].value_counts())
else:
    print(f"Directory not found: {audioset_dir} — skipping AudioSet.")